# ✏️ TOONIFY — Sketch LoRA Training with Hugging Face AutoTrain

Train a custom **Stable Diffusion 1.5 LoRA** using official **AutoTrain Advanced**.

| Setting | Value |
|---------|-------|
| Base Model | `runwayml/stable-diffusion-v1-5` |
| Resolution | 512×512 |
| Trigger Word | `abhi_sketch_style` |
| Output | `sketch_lora.safetensors` |

> **Runtime → Change runtime type → T4 GPU** before running!

## Step 1: Install AutoTrain Advanced

In [ ]:
!pip install -q autotrain-advanced
!autotrain setup
print("✅ AutoTrain environment configured!")

## Step 2: Mount Google Drive & Prepare Dataset

In [ ]:
import os
import zipfile
import shutil
from google.colab import drive

if os.path.exists('/content/drive'):
    print('Google Drive already mounted.')
else:
    drive.mount('/content/drive')

TRIGGER_WORD = 'abhi_sketch_style'
autotrain_dataset_dir = '/content/train_data'
os.makedirs(autotrain_dataset_dir, exist_ok=True)

drive_folder = '/content/drive/MyDrive/sketch_dataset'
zip_path_drive = '/content/drive/MyDrive/sketch_dataset.zip'
zip_path_local = '/content/sketch_dataset.zip'

if os.path.exists(drive_folder):
    print('✅ Found sketch_dataset folder in Google Drive!')
    for root, dirs, files in os.walk(drive_folder):
        for file in files:
            if file.endswith(('.png', '.jpg', '.jpeg', '.txt')):
                shutil.copy2(os.path.join(root, file), os.path.join(autotrain_dataset_dir, file))
elif os.path.exists(zip_path_drive) or os.path.exists(zip_path_local):
    zip_target = zip_path_drive if os.path.exists(zip_path_drive) else zip_path_local
    with zipfile.ZipFile(zip_target, 'r') as z:
        z.extractall(autotrain_dataset_dir)

import glob
images = glob.glob(f'{autotrain_dataset_dir}/*.png') + glob.glob(f'{autotrain_dataset_dir}/*.jpg')
print(f'✅ Found {len(images)} images ready for AutoTrain!')

## Step 3: Run AutoTrain Web GUI (1 Click)

In [ ]:
# Launch official AutoTrain GUI
!autotrain app --host 0.0.0.0 --port 7860 --share

## Step 4: Save LoRA to Google Drive

In [ ]:
import shutil
import os, glob

gdrive_output = '/content/drive/MyDrive/models'
os.makedirs(gdrive_output, exist_ok=True)

files = glob.glob('/content/**/*.safetensors', recursive=True)
lora_files = [f for f in files if 'v1-5' not in f]

if lora_files:
    latest = lora_files[0]
    dest = os.path.join(gdrive_output, 'sketch_lora.safetensors')
    shutil.copy2(latest, dest)
    print(f'✅ Saved {os.path.basename(latest)} to Google Drive: {dest}')
else:
    print('⚠️ Check AutoTrain output folder for generated .safetensors weights.')